# 🤖🔍 Intelligent SQL Agent: Automating Database Queries with Azure OpenAI & LangChain

Welcome! In this notebook, you'll build an intelligent agent that can answer questions about a SQL database. Leveraging Azure OpenAI, LangChain, and Langgraph, the agent will:

- Discover available tables in the database
- Identify which tables are relevant to a user’s question
- Retrieve schemas for those tables
- Generate and validate SQL queries using LLMs
- Execute queries and handle errors automatically
- Return results and formulate clear responses

This hands-on tutorial demonstrates how AI-powered agents can automate complex database interactions, making data access seamless and error-free.

In [ ]:
# %pip install --quiet python-dotenv
# %pip install --quiet langchain
# %pip install --quiet langchain-core
# %pip install --quiet langchain-openai
# %pip install --quiet langchain-community
# %pip install --quiet langgraph
# %pip install --quiet requests
# %pip install --quiet typing-extensions

# Import Packages

In [ ]:
import sys
import os
import requests
from pprint import pprint
from langchain_core.messages import AIMessage
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langgraph.graph import END, StateGraph, START, MessagesState
from langgraph.prebuilt import create_react_agent, ToolNode
from langchain_community.utilities import SQLDatabase
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# # check the version name
# import importlib.metadata
# try:
#     version = importlib.metadata.version('langchain_core')
#     print(f"LangChain Core version: {version}")
# except importlib.metadata.PackageNotFoundError:
#     print("langchain_core is not installed")

# Azure OpenAI Chat Model Integration

This example demonstrates how to use the `AzureChatOpenAI` class from the `langchain_openai` package to interact with Azure OpenAI's GPT models.

#### Key Steps:
1. **Setup**: Import necessary libraries and retrieve credentials for the Azure OpenAI API.
2. **Credentials**: Ensure the Azure API key and endpoint are correctly set.
3. **Error Handling**: Check if the API key and endpoint are missing, and raise errors if needed.
4. **Chat Model**: Initialize the `AzureChatOpenAI` model by providing required credentials like API key, endpoint, model type, and API version.

#### Notes:
- Ensure that Azure OpenAI is set up and environment variables for the API key and endpoint are configured.


In [ ]:
import init_creds as creds
sys.path.insert(1, '../../')

In [ ]:
#Configure the azure openai settings

# Configure Azure OpenAI And OpenAI Embeddings
# AZURE_OPENAI_KEY = os.getenv('OPENAI_API_KEY')
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
# AZURE_OPENAI_API_VERSION = os.getenv('OPENAI_API_VERSION')

# Configure Azure OpenAI And OpenAI Embeddings
AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

In [ ]:
# Checking
llm.invoke("Hello")

# Configure the database
In this tutorial, we will be connecting to the SQLite database. SQLite is a lightweight database that is easy to set up and use. We will be loading the chinook database, which is a sample database that represents a digital media store. Find more information about the database [here](https://www.sqlitetutorial.net/sqlite-sample-database/).

In [ ]:
#download the Chinook database
url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"

response = requests.get(url)

if response.status_code == 200:
    # Open a local file in binary write mode
    with open("Chinook.db", "wb") as file:
        # Write the content of the response (the file) to the local file
        file.write(response.content)
    print("File downloaded and saved as Chinook.db")
else:
    print(f"Failed to download the file. Status code: {response.status_code}")

In [ ]:
# Connect to the database using langchain SQL wrapper

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

# Tools for database interactions
`langchain-community` implements some built-in tools for interacting with our SQLDatabase, including tools for listing tables, reading table schemas, and checking and running queries

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# Create the toolkit and add it to an agent
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

# Using Pre-Built Agent

**ReAct**: 
- A ReAct agent in LangGraph is an AI agent that implements the "Reasoning and Acting" (ReAct) pattern. This pattern enables the agent to iteratively reason about a user query, determine necessary actions, execute those actions (often by calling tools), and then observe the results to inform further reasoning or generate a final answer.

- ReAct agent uses the “reasoning and acting” framework to combine chain of thought (CoT) reasoning with external tool use. The ReAct framework enhances the ability of a large language model (LLM) to handle complex tasks and decision-making in agentic workflows.

In [ ]:
from langchain.agents import create_agent

system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

# Create the agent using create_agent
agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
)

In [ ]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

# Customizing the Agent

- While the prebuilt agent is a great way to get started quickly, it gives the agent access to all available tools at every step. Earlier, we relied on the system prompt to guide its behavior—for example, we instructed it to always begin by listing tables and to check queries before execution.

- As you progress, I want you to see how LangGraph allows us to enforce even greater control by customizing the agent’s workflow. In the next section, we’ll build a simple ReAct-agent setup with dedicated nodes for specific tool calls. We’ll use the same state as the prebuilt agent, but this approach will help you understand how to structure agentic workflows for more predictable

We construct dedicated nodes for the following steps:

- Listing DB tables
- Calling the "get schema" tool
- Generating a query
- Checking the query
- Putting these steps in dedicated nodes lets us (1) force tool-calls when needed, and (2) customize the prompts associated with each step.

In [ ]:
# Select the schema tool and create a node for fetching table schemas
get_schema_tool = next(tool for tool in tools if tool.name == "sql_db_schema")
get_schema_node = ToolNode([get_schema_tool], name="get_schema")

# Select the query tool and create a node for running queries
run_query_tool = next(tool for tool in tools if tool.name == "sql_db_query")
run_query_node = ToolNode([run_query_tool], name="run_query")

## Define Tools

This section provides functions for each step in the SQL agent workflow. Each function uses LangChain's tool binding and invocation features to control how the agent interacts with the database.

- **`list_tables(state)`**  
  Manually creates a tool call to list all tables in the database.  
  Finds the correct tool, invokes it, and returns the result as a message.

- **`call_get_schema(state)`**  
  Forces the LLM to call the schema tool using `.bind_tools()`.  
  Invokes the tool with the current state messages and returns the response.

- **`generate_query(state)`**  
  Adds a system prompt to guide the LLM in generating a SQL query.  
  Binds the LLM to the query tool and invokes it with the prompt and messages.

- **`check_query(state)`**  
  Validates the generated SQL query for errors.  
  Prepares a system prompt and user message, binds the LLM to the query tool, and invokes it for validation.

---

**Key Concepts:**  

- **Tool binding** restricts the LLM to only use specified tools at each step, ensuring the agent follows the intended workflow and only accesses relevant capabilities.
- **Tool invocation** runs the tool with the provided arguments and returns results as messages, allowing the agent to interact with the database in a controlled manner.
- **State (`MessagesState`)** is a data structure that tracks the entire conversation and all agent actions.  
  - It holds all messages exchanged (user, system, tool calls, and responses).
  - This stateful approach is essential for agentic workflows because each step may depend on previous actions and context.
  - By passing `state` between functions, the agent maintains continuity, can reference earlier queries, and supports iterative reasoning and error handling.
- This modular approach ensures reliability, traceability, and control in the agent workflow, making it easier to debug, extend, and enforce business logic.

In [ ]:
# Example: create a predetermined tool call
def list_tables(state: MessagesState):
    """List all tables in the database.
    
    class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

    Args:
        state (MessagesState): The current state of the conversation.

        Returns:
        dict: A dictionary with a "messages" key containing a list of AIMessage objects representing the tool call, tool response, and final response.
    """
    tool_call = {
        "name": "sql_db_list_tables",
        "args": {},
        "id": "abc123",
        "type": "tool_call",
    }
    tool_call_message = AIMessage(content="", tool_calls=[tool_call])

    list_tables_tool = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    tool_message = list_tables_tool.invoke(tool_call)
    response = AIMessage(f"Available tables: {tool_message.content}")

    return {"messages": [tool_call_message, tool_message, response]}

# Example: force a model to create a tool call
def call_get_schema(state: MessagesState):
    # Note that LangChain enforces that all models accept `tool_choice="any"`
    # as well as `tool_choice=<string name of tool>`.
    llm_with_tools = llm.bind_tools([get_schema_tool], tool_choice="any")
    response = llm_with_tools.invoke(state["messages"])

    return {"messages": [response]}

In [ ]:
generate_query_system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
""".format(
    dialect=db.dialect,
    top_k=5,
)

def generate_query(state: MessagesState):
    # Prepare a system prompt to guide the LLM for SQL query generation
    system_message = {
        "role": "system",
        "content": generate_query_system_prompt,
    }
    # Bind the LLM to only use the run_query_tool for this step
    # No forced tool call, so the model can respond naturally
    llm_with_tools = llm.bind_tools([run_query_tool])
    response = llm_with_tools.invoke([system_message] + state["messages"])

    # Return the generated query message
    return {"messages": [response]}

In [ ]:
check_query_system_prompt = """
You are a SQL expert with a strong attention to detail.
Double check the {dialect} query for common mistakes, including:
- Using NOT IN with NULL values
- Using UNION when UNION ALL should have been used
- Using BETWEEN for exclusive ranges
- Data type mismatch in predicates
- Properly quoting identifiers
- Using the correct number of arguments for functions
- Casting to the correct data type
- Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes,
just reproduce the original query.

You will call the appropriate tool to execute the query after running this check.
""".format(dialect=db.dialect)

def check_query(state: MessagesState):
    # Prepare system prompt for query validation
    system_message = {
        "role": "system",
        "content": check_query_system_prompt,
    }

    # Extract query from last tool call
    tool_call = state["messages"][-1].tool_calls[0]
    user_message = {"role": "user", "content": tool_call["args"]["query"]}

    # Bind LLM to only use run_query_tool
    llm_with_tools = llm.bind_tools([run_query_tool], tool_choice="any")
    response = llm_with_tools.invoke([system_message, user_message])
    response.id = state["messages"][-1].id

    return {"messages": [response]}

# SQL Agent Workflow Graph Setup

In this section, we construct a state graph to orchestrate the workflow of our SQL agent, ensuring each step in the query process is handled in a controlled and logical sequence.

#### Key Components:

1. **State Graph Initialization**:
   - The `StateGraph` is initialized with `MessagesState`, which maintains the conversational context and agent state throughout the workflow.

2. **Node Addition**:
   - **Nodes Added**:
     - **list_tables**: Initiates the process by listing all available tables in the database.
     - **call_get_schema**: Triggers the retrieval of relevant table schemas.
     - **get_schema_node**: Executes the schema-fetching tool.
     - **generate_query**: Uses the LLM to generate a SQL query based on the user’s question and schema information.
     - **check_query**: Validates the generated query for common SQL mistakes.
     - **run_query_node**: Executes the validated query against the database.

3. **Edge Configuration**:
   - Edges define the flow between nodes:
     - The workflow starts at `START` and proceeds to `list_tables`.
     - Each subsequent node is connected in the logical order of the SQL agent’s reasoning process.
     - After running a query, the flow returns to `generate_query` to handle errors or refine the query if needed.

4. **Conditional Edges**:
   - Conditional logic is applied after query generation, using `should_continue` to determine whether to proceed to query validation or loop back for further refinement.

This graph-based setup provides a clear, modular structure for the SQL agent, enabling precise control over each step and facilitating robust error handling and iterative improvement of queries.

In [ ]:
# Determines whether the workflow should continue to query checking or end.
# If the last message does not contain any tool calls, the workflow ends.
# Otherwise, it proceeds to the query checking step.
def should_continue(state: MessagesState) -> Literal[END, "check_query"]:
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return END
    else:
        return "check_query"

In [ ]:
# Build the SQL Agent workflow graph

builder = StateGraph(MessagesState)  # Initialize the state graph with message state

# Add nodes for each step in the workflow
builder.add_node(list_tables)                # Node to list all tables in the database
builder.add_node(call_get_schema)            # Node to call the schema tool
builder.add_node(get_schema_node, "get_schema")  # Node to fetch table schemas
builder.add_node(generate_query)             # Node to generate a SQL query
builder.add_node(check_query)                # Node to validate the query
builder.add_node(run_query_node, "run_query")    # Node to execute the query

# Define the edges (flow) between nodes
builder.add_edge(START, "list_tables")           # Start by listing tables
builder.add_edge("list_tables", "call_get_schema")   # Then get relevant schemas
builder.add_edge("call_get_schema", "get_schema")    # Fetch schema details
builder.add_edge("get_schema", "generate_query")     # Generate query based on schema

# Add conditional edges to handle query refinement or validation
builder.add_conditional_edges(
    "generate_query",
    should_continue,  # Decide whether to check query or refine it further
)

builder.add_edge("check_query", "run_query")         # After checking, run the query
builder.add_edge("run_query", "generate_query")      # If errors, loop back to query generation

# Compile the agent graph for execution
agent = builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = "Which genre on average has the longest tracks?"

# Collect the final response from the agent stream
final_response = None
for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    final_response = step["messages"][-1]  # Overwrite until last step

# Display or save the final response
if final_response is not None:
    print(final_response.content)  # This prints only the final answer
else:
    print("No response received from the agent.")